# BMW Sales Data: Cleaning & Feature Engineering

This notebook contains the data cleaning, structural validation, and feature engineering steps for the BMW Sales Dataset (2024-2025). The output of this notebook is the cleaned dataset `bmw_cleaned.csv` which will be used in subsequent phases for PostgreSQL and Power BI reporting.

In [1]:
import pandas as pd
import numpy as np

## 1. Load Dataset

In [2]:
df = pd.read_csv('../Dataset/bmw_sales_2024_2025.csv')
print(f"Dataset Shape: {df.shape}")
df.head(3)

## 2. Structural Validation

- Check for duplicate `transaction_id` values.
- Parse `sale_date` to datetime and verify it is consistent with `sale_year`, `sale_month`, and `sale_quarter`.

In [3]:
dup_ids = df['transaction_id'].duplicated().sum()
print(f"Duplicate transaction_ids: {dup_ids}")

In [4]:
# Date consistency check
df['sale_date_parsed'] = pd.to_datetime(df['sale_date'])

expected_years = df['sale_date_parsed'].dt.year
expected_months = df['sale_date_parsed'].dt.month
expected_quarters = 'Q' + ((expected_months - 1) // 3 + 1).astype(str)

year_mismatch = (expected_years != df['sale_year']).sum()
month_mismatch = (expected_months != df['sale_month']).sum()
quarter_mismatch = (expected_quarters != df['sale_quarter']).sum()

print(f"Year mismatches: {year_mismatch}")
print(f"Month mismatches: {month_mismatch}")
print(f"Quarter mismatches: {quarter_mismatch}")
print(f"Number of Date Mismatches: {year_mismatch + month_mismatch + quarter_mismatch}")

## 3. Business Rule Validation

- Check if `final_sale_price_usd <= 0` or `msrp_usd <= 0`.
- Check if `discount_percent` is outside the range `[0, 100]`.

In [ ]:
price_check = df[(df['final_sale_price_usd'] <= 0) | (df['msrp_usd'] <= 0)]
discount_check = df[(df['discount_percent'] < 0) | (df['discount_percent'] > 100)]

print(f"Rows with final_sale_price_usd <= 0: {len(price_check)}")
print(f"Rows with msrp_usd <= 0: {len(price_check)}")
print(f"Rows with discount_percent outside [0, 100]: {len(discount_check)}") 

## 4. Missing Value Handling

Per instructions, we must apply specific business rules for missing values:
- `loan_term_months`: **Leave as NULL.** Do not fill with 0 or mean. (Reason: cash/subscription customers have no loan).
- `customer_satisfaction_score`: **Leave as NULL.** Do not impute. (Reason: missing means no survey response).
- Check for other missing values.

In [6]:
missing_values = df.isnull().sum()
print("Missing values count per column:")
print(missing_values[missing_values > 0])

## 5. Feature Engineering

We add the following four features to the dataset:
1. `discount_bucket` (Low (0–5%) / Medium (5–10%) / High (10–20%) / Very High (20%+))
2. `year_month` (formatted period `sale_date.dt.to_period('M')` for monthly groupings)
3. `delivery_category` (Fast (<7 days) / Normal (7–14 days) / Delayed (>14 days))

In [ ]:

# 1. discount_bucket
df['discount_bucket'] = pd.cut(df['discount_percent'], bins=[0, 5, 10, 20, 100], labels=['Low', 'Medium', 'High', 'Very High'], include_lowest=True)

# 2. year_month
df['year_month'] = df['sale_date_parsed'].dt.to_period('M')

# 3. delivery_category
df['delivery_category'] = np.select(
    [
        df['delivery_days'] < 7,
        (df['delivery_days'] >= 7) & (df['delivery_days'] <= 14),
        df['delivery_days'] > 14
    ],
    ['Fast', 'Normal', 'Delayed'],
    default='Unknown'
)

print("New feature columns verified:")
print(f"  profit_proxy: min={df['profit_proxy'].min():.2f}, mean={df['profit_proxy'].mean():.2f}, max={df['profit_proxy'].max():.2f}")
print("  discount_bucket distribution:")
print(df['discount_bucket'].value_counts())
print("  delivery_category distribution:")
print(df['delivery_category'].value_counts())

## 6. Export Cleaned Dataset

We drop the intermediate parsed date helper column and export the clean dataset to `bmw_cleaned.csv`.

In [8]:
df_clean = df.drop(columns=['sale_date_parsed'])
df_clean.to_csv('../bmw_cleaned.csv', index=False)
print(f"Exported cleaned dataset with shape: {df_clean.shape}")